# Imports

In [1]:
# I'm using a conda environment with python 3.12.7
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

## The data

In [2]:
# Load dataset
data = pd.read_csv("weather_data.csv")

# Inspect dataset
data.head()

,prec,temp_max,temp_min,wind,weather
0,10.9,10.6,2.8,4.5,rain
1,0.8,11.7,7.2,2.3,rain
2,20.3,12.2,5.6,4.7,rain
3,1.3,8.9,2.8,6.1,rain
4,2.5,4.4,2.2,2.2,rain


# Splitting features (x) and target (y)

In [3]:
# Features: prec, temp_max, temp_min, wind
X = data[["prec", "temp_max", "temp_min", "wind"]]

# Target: weather (rain or sun)
y = data["weather"]

# Train-test split (80% train, 20% test) with stratification to balance rain/sun
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
# Count occurrences of each class in the target variable to verify balance
print("Training set class distribution:\n", y_train.value_counts())

Training set class distribution:
 weather
sun     512
rain    512
Name: count, dtype: int64


In [4]:
# Inspect the features
X_train.head()

,prec,temp_max,temp_min,wind
455,0.0,23.9,12.8,3.4
883,0.0,30.0,11.7,1.8
1096,0.0,27.8,12.2,2.1
129,6.6,20.0,12.8,3.7
1082,4.3,15.6,10.6,3.3


In [5]:
# Inspect the target
y_train.head()

455      sun
883      sun
1096     sun
129     rain
1082    rain
Name: weather, dtype: object

# Classification models

Scaling with Min-Max scaling

In [6]:
#Scale features using Min-Max scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### K-Nearest Neighbour (K-NN) with k=1,3,5,15

In [7]:
for k in [1, 3, 5, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    
    print(f"\nKNN (k={k})")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



KNN (k=1)
Classification Report:
               precision    recall  f1-score   support

        rain       0.80      0.79      0.79       129
         sun       0.79      0.80      0.79       128

    accuracy                           0.79       257
   macro avg       0.79      0.79      0.79       257
weighted avg       0.79      0.79      0.79       257

Confusion Matrix:
 [[102  27]
 [ 26 102]]

KNN (k=3)
Classification Report:
               precision    recall  f1-score   support

        rain       0.83      0.84      0.83       129
         sun       0.83      0.83      0.83       128

    accuracy                           0.83       257
   macro avg       0.83      0.83      0.83       257
weighted avg       0.83      0.83      0.83       257

Confusion Matrix:
 [[108  21]
 [ 22 106]]

KNN (k=5)
Classification Report:
               precision    recall  f1-score   support

        rain       0.84      0.83      0.83       129
         sun       0.83      0.84      0.83     

#### Evaluation
The best performing KNN models are with k=3 or k=5, both giving 83% accuracy with balanced precision/recall. Between them, either is fine, but k=3 might be slightly preferable as it handles small variations better.

### Logistic Regression

In [8]:
# With scaled data
logreg_scaled = LogisticRegression(max_iter=200)
logreg_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = logreg_scaled.predict(X_test_scaled)

print("\nLogistic Regression (scaled features)")
print("Classification Report:\n", classification_report(y_test, y_pred_scaled))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_scaled))

# With non-scaled data
logreg = LogisticRegression(max_iter=200)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("\nLogistic Regression (non-scaled features)")
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Logistic Regression (scaled features)
Classification Report:
               precision    recall  f1-score   support

        rain       0.82      0.78      0.80       129
         sun       0.79      0.83      0.81       128

    accuracy                           0.80       257
   macro avg       0.80      0.80      0.80       257
weighted avg       0.80      0.80      0.80       257

Confusion Matrix:
 [[100  29]
 [ 22 106]]

Logistic Regression (non-scaled features)
Classification Report:
               precision    recall  f1-score   support

        rain       1.00      0.89      0.94       129
         sun       0.90      1.00      0.95       128

    accuracy                           0.95       257
   macro avg       0.95      0.95      0.95       257
weighted avg       0.95      0.95      0.95       257

Confusion Matrix:
 [[115  14]
 [  0 128]]


#### Evaluation
Logistic Regression performed best for this dataset, especially without scaling, where it achieved 95% accuracy and near-perfect precision/recall balance. Compared to KNN, which reached around 83% accuracy, Logistic Regression proved more effective at separating the classes and made far fewer classification errors. This suggests that the dataset’s features provide good linear separability, making Logistic Regression the most suitable method.